In [6]:
from collections import defaultdict
from functools import reduce

Célula 1/1.1 | Upload do dataset

In [1]:
#Para carregar o documento direto do computadr

from google.colab import files
uploaded = files.upload()  # selecione o nyc_tripdata_2024_sample_1M.csv
csv_original = list(uploaded.keys())[0]

KeyboardInterrupt: 

In [3]:
#Direto do Colab

from google.colab import drive
drive.mount('/content/drive')
csv_original = '/content/drive/MyDrive/5º Semestre/Processamento de Dados Massivos - 5º Semestre/Trabalho 1 /nyc_tripdata_2024_sample_1M.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Célula 2 | Conferindo o cabeçalho (mapeamento de colunas)

In [5]:
with open(csv_original, 'r') as f:
    cabecalho = f.readline()
print(cabecalho)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,arquivo_origem



Célula 3 | Infraestrutura MapReduce (Funções genéricas de Shuffle e Reduce

Estas duas funções serão reaproveitadas em todas as questões do trabalho, seguindo o mesmo
padrão MapReduce visto em aula: o mapper muda a cada pergunta, mas shuffle (agrupamento
por chave) e reducer (agregação) são genéricos.)

In [10]:
def shuffle(nome_arquivo_entrada, nome_arquivo_saida):
    resultado_intermediario = defaultdict(list)
    with open(nome_arquivo_entrada, 'r') as arquivo_entrada:
        for linha in arquivo_entrada:
            chave, valor = linha.strip('\n').split('\t')
            resultado_intermediario[chave].append(valor)
    resultado = dict(resultado_intermediario)
    with open(nome_arquivo_saida, 'w') as arquivo_shuffle:
        arquivo_shuffle.write(str(resultado))

def reducer(nome_arquivo_entrada, nome_arquivo_saida, funcao_reduce, funcao_parse=float, valor_inicial=None):
    with open(nome_arquivo_entrada, 'r') as arquivo:
        dict_entrada = eval(arquivo.read())
    dict_result = {}
    for chave, lista in dict_entrada.items():
        lista_convertida = [funcao_parse(v) for v in lista] if funcao_parse else lista
        if valor_inicial is not None:
            dict_result[chave] = reduce(funcao_reduce, lista_convertida, valor_inicial)
        else:
            dict_result[chave] = reduce(funcao_reduce, lista_convertida)
    with open(nome_arquivo_saida, 'w') as arquivo_reduce:
        for item in dict_result:
            arquivo_reduce.write(f'{item}\t{dict_result[item]}\n')

def somar(x, y):
    return x + y

Questão 1 | Número de viagens por tipo de pagamento

In [13]:
payment_type_dict = {
    '0': 'Flex Fare trip',
    '1': 'Credit card',
    '2': 'Cash',
    '3': 'No charge',
    '4': 'Dispute',
    '5': 'Unknown',
    '6': 'Voided trip',
}

def mapper_viagens_por_pagamento(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, 'r') as arquivo_entrada, open(nome_arquivo_saida, 'w') as arquivo_saida:
        cabecalho = arquivo_entrada.readline()
        for linha in arquivo_entrada:
            campos = linha.strip('\n').split(',')
            tipo_pagamento = payment_type_dict.get(campos[9], 'Desconhecido')
            arquivo_saida.write(f'{tipo_pagamento}\t1\n')

mapper_viagens_por_pagamento(csv_original, 'map_q1.txt')
shuffle('map_q1.txt', 'shuffle_q1.txt')
reducer('shuffle_q1.txt', 'resultado_q1.txt', somar, funcao_parse=int)

In [ ]:
with open('resultado_q1.txt', 'r') as arquivo:
    print(arquivo.read())

Questão 2 | Receita total por tipo de pagamento

In [15]:
def mapper_receita_por_pagamento(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, 'r') as arquivo_entrada, open(nome_arquivo_saida, 'w') as arquivo_saida:
        cabecalho = arquivo_entrada.readline()
        for linha in arquivo_entrada:
            campos = linha.strip('\n').split(',')
            tipo_pagamento = payment_type_dict.get(campos[9], 'Desconhecido')
            total_amount = campos[16]
            arquivo_saida.write(f'{tipo_pagamento}\t{total_amount}\n')

mapper_receita_por_pagamento(csv_original, 'map_q2.txt')
shuffle('map_q2.txt', 'shuffle_q2.txt')
reducer('shuffle_q2.txt', 'resultado_q2.txt', somar, funcao_parse=float)

In [ ]:
with open('resultado_q2.txt', 'r') as arquivo:
    print(arquivo.read())

**Observação sobre a Q2:** o dataset contém 15.571 viagens com "fare_amount"/"total_amount"
negativos, provavelmente referentes a estornos ou ajustes. Eles não foram filtrados, então a
"receita total" apresentada já é líquida desses ajustes.

Questão 3 | Tarifa média cobrada nas viagens

In [16]:
def mapper_tarifas(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, 'r') as arquivo_entrada, open(nome_arquivo_saida, 'w') as arquivo_saida:
        cabecalho = arquivo_entrada.readline()
        for linha in arquivo_entrada:
            campos = linha.strip('\n').split(',')
            fare_amount = campos[10]
            arquivo_saida.write(f'geral\t{fare_amount}\n')

mapper_tarifas(csv_original, 'map_q3.txt')
shuffle('map_q3.txt', 'shuffle_q3.txt')

with open('shuffle_q3.txt', 'r') as arquivo:
    dict_tarifas = eval(arquivo.read())

lista_tarifas = [float(v) for v in dict_tarifas['geral']]
tarifa_media = reduce(somar, lista_tarifas) / len(lista_tarifas)

with open('resultado_q3.txt', 'w') as arquivo:
    arquivo.write(f'tarifa_media\t{tarifa_media:.2f}\n')

In [ ]:
with open('resultado_q3.txt', 'r') as arquivo:
    print(arquivo.read())

Questão 4 | Data e hora da viagem mais longa

In [17]:
def mapper_viagem_mais_longa(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, 'r') as arquivo_entrada, open(nome_arquivo_saida, 'w') as arquivo_saida:
        cabecalho = arquivo_entrada.readline()
        for linha in arquivo_entrada:
            campos = linha.strip('\n').split(',')
            trip_distance = campos[4]
            pickup = campos[1]
            dropoff = campos[2]
            arquivo_saida.write(f'viagem\t{trip_distance}|{pickup}|{dropoff}\n')

def maior_viagem(a, b):
    distancia_a = float(a.split('|')[0])
    distancia_b = float(b.split('|')[0])
    return a if distancia_a >= distancia_b else b

mapper_viagem_mais_longa(csv_original, 'map_q4.txt')
shuffle('map_q4.txt', 'shuffle_q4.txt')
reducer('shuffle_q4.txt', 'resultado_q4.txt', maior_viagem, funcao_parse=None)

In [ ]:
with open('resultado_q4.txt', 'r') as arquivo:
    print(arquivo.read())

**Observação sobre a Q4:** o resultado aponta uma viagem de ~86.789 milhas em 13 minutos,
o que é fisicamente impossível (implicaria uma velocidade de centenas de milhares de km/h).
Uma checagem no dataset completo mostra que existem 23 viagens acima de 200 milhas, valor já
inviável para viagens de táxi dentro de Nova York — provavelmente erro de sensor/GPS na coleta
do dado original.

Isso evidencia uma limitação do MapReduce puro (e de pipelines de Big Data em geral): ele
processa e agrega os dados exatamente como estão, sem qualquer tratamento de qualidade/outliers
embutido. Resolver isso exigiria uma etapa de limpeza | por exemplo, filtrar "trip_distance"
acima de um limite plausível | **antes** do mapper, e não depois do resultado pronto.

Questão 5 | Quantidade de viagens por hora

In [18]:
def mapper_viagens_por_hora(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, 'r') as arquivo_entrada, open(nome_arquivo_saida, 'w') as arquivo_saida:
        cabecalho = arquivo_entrada.readline()
        for linha in arquivo_entrada:
            campos = linha.strip('\n').split(',')
            hora = campos[1][11:13]
            arquivo_saida.write(f'{hora}\t1\n')

mapper_viagens_por_hora(csv_original, 'map_q5.txt')
shuffle('map_q5.txt', 'shuffle_q5.txt')
reducer('shuffle_q5.txt', 'resultado_q5.txt', somar, funcao_parse=int)

In [ ]:
with open('resultado_q5.txt', 'r') as arquivo:
    print(arquivo.read())

Questão 6 | Distância total percorrida por hora

In [19]:
def mapper_distancia_por_hora(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, 'r') as arquivo_entrada, open(nome_arquivo_saida, 'w') as arquivo_saida:
        cabecalho = arquivo_entrada.readline()
        for linha in arquivo_entrada:
            campos = linha.strip('\n').split(',')
            hora = campos[1][11:13]
            trip_distance = campos[4]
            arquivo_saida.write(f'{hora}\t{trip_distance}\n')

mapper_distancia_por_hora(csv_original, 'map_q6.txt')
shuffle('map_q6.txt', 'shuffle_q6.txt')
reducer('shuffle_q6.txt', 'resultado_q6.txt', somar, funcao_parse=float)

In [ ]:
with open('resultado_q6.txt', 'r') as arquivo:
    print(arquivo.read())

**Observação sobre a Q6:** como visto na Q4, o dataset contém algumas viagens com "trip_distance"
implausível (23 viagens acima de 200 milhas). Esses valores entram na soma por hora sem qualquer
filtro, o que pode inflar levemente o total de algumas horas específicas.

Resultados | Verificando as saídas

In [20]:
arquivos_resultado = {
    'Q1 - Viagens por tipo de pagamento': 'resultado_q1.txt',
    'Q2 - Receita total por tipo de pagamento': 'resultado_q2.txt',
    'Q3 - Tarifa média': 'resultado_q3.txt',
    'Q4 - Viagem mais longa': 'resultado_q4.txt',
    'Q5 - Viagens por hora': 'resultado_q5.txt',
    'Q6 - Distância total por hora': 'resultado_q6.txt',
}

for titulo, nome_arquivo in arquivos_resultado.items():
    print(f'--- {titulo} ---')
    with open(nome_arquivo, 'r') as arquivo:
        print(arquivo.read())

--- Q1 - Viagens por tipo de pagamento ---
Cash	136221
Credit card	743405
Flex Fare trip	97124
Dispute	16543
No charge	6707

--- Q2 - Receita total por tipo de pagamento ---
Cash	3168095.899999832
Credit card	21785219.95001744
Flex Fare trip	2376069.7700001337
Dispute	25214.50999999983
No charge	53932.4799999999

--- Q3 - Tarifa média ---
tarifa_media	18.86

--- Q4 - Viagem mais longa ---
viagem	86789.2|2024-05-10 17:33:00|2024-05-10 17:46:00

--- Q5 - Viagens por hora ---
14	59345
11	48295
02	12280
09	42309
13	55362
18	71403
19	62752
07	28065
08	38308
23	42363
21	58333
17	67880
15	60205
01	18822
22	54581
10	44804
12	53128
20	56542
05	6194
00	29165
16	61563
04	6054
06	13966
03	8281

--- Q6 - Distância total por hora ---
14	215732.26999999877
11	145189.3000000011
02	36643.430000000124
09	222652.02000000176
13	190847.82999999996
18	252601.1899999956
19	265021.9799999992
07	195237.06999999928
08	169111.7300000016
23	161753.44999999925
21	283775.0800000024
17	321659.2100000004
15	312374.37